# Approche 4 — `PerformerMessagePassingFunction` (Item 4)

Quatrième notebook d'attention pour l'équipe R&D RTE. Il déroule de bout en bout la `PerformerMessagePassingFunction` (attention linéaire all-to-all, forme kernel-trick) sur le serveur GPU de La Javaness, dans le même pipeline `RecurrentCoupler` que les Approches 1 à 3.

**Prérequis :**
- `uv sync --extra gpu`, `pypowsybl==1.15.0` et `pypowsybl-to-energnn` installés.
- Notebooks 00 à 03 lus au préalable.

**Durée attendue :** 7 à 12 minutes sur le serveur GPU de La Javaness.


## 1. Mise en place de l'environnement

Imports : JAX, NumPy, Flax NNX, optax. Vérification du device JAX et du dtype par défaut. On attend `CudaDevice` (serveur GPU de La Javaness) et `float32`.


In [1]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root / "benchmarks"))

print(f"JAX version:    {jax.__version__}")
print(f"JAX devices:    {jax.devices()}")
print(f"Default dtype:  {jnp.array(0.0).dtype}")
print(f"Repo root:      {_root}")


JAX version:    0.9.0


JAX devices:    [CudaDevice(id=0), CudaDevice(id=1)]
Default dtype:  float32
Repo root:      /home/van.tuan/GNN/energnn_attention


## 2. Méthode — `PerformerMessagePassingFunction`

Pour l'Item 4 du backlog (sec 3.4), le mécanisme est une attention linéaire **all-to-all** sur les adresses, sans softmax et sans noyau à features aléatoires. À la différence de GATv2 et MultiHeadQKV qui agrègent sur les voisins du récepteur via les hyper-edges, le Performer agrège sur **toutes** les adresses (le port_dict de la connectivité n'est pas consulté dans cette couche).

À chaque adresse $a$ on calcule trois projections par MLP partagées:

$$
Q_a = Q_\theta(h_a) \in \mathbb{R}^{d_{QK}}, \qquad
K_{a'} = K_\theta(h_{a'}) \in \mathbb{R}^{d_{QK}}, \qquad
V_{a'} = V_\theta(h_{a'}) \in \mathbb{R}^{d_V}.
$$

La forme littérale du spec (Form A) est la somme bilinéaire « per-récepteur »:

$$
\psi_\theta(h, x)_a = \sigma\!\left(
    \frac{1}{\sqrt{d_{QK}}}
    \sum_{a' \in \mathcal{A}_x^{\mathrm{real}}}
    (K_{a'}^\top Q_a)\, V_{a'}
\right),
$$

avec $\mathcal{A}_x^{\mathrm{real}}$ = adresses non fictives (le padding est masqué avant la somme), et $\sigma$ = `outer_activation` appliquée après agrégation.

La reformulation « kernel trick » donnée dans le spec (Form B) factorise la somme:

$$
\psi_\theta(h, x)_a = \sigma\!\left(
    \frac{1}{\sqrt{d_{QK}}}
    \Bigl(\sum_{a'} V_{a'} K_{a'}^\top\Bigr) Q_a
\right).
$$

L'implementation calcule la matrice $M = \sum_{a'} V_{a'} K_{a'}^\top \in \mathbb{R}^{d_V \times d_{QK}}$ une seule fois, puis multiplie par $Q_a$ par récepteur. Coût total: $\mathcal{O}(n_{\mathrm{addr}} \cdot d_V \cdot d_{QK})$ au lieu de $\mathcal{O}(n_{\mathrm{addr}}^2 \cdot d_V)$ pour la forme naïve.

Le facteur $1/\sqrt{d_{QK}}$ suit la convention de Vaswani et al. 2017 (variance du score $K^\top Q$ bornée en $\mathcal{O}(1)$ si $Q, K$ i.i.d.). Le flag `scale_scores=True` par défaut; mettre `False` reproduit la formule littérale du backlog.

Références:
- Katharopoulos et al. « Transformers are RNNs » (ICML 2020) — forme linéaire sans softmax, base du kernel-trick utilisé ici.
- Vaswani et al. « Attention Is All You Need » (NeurIPS 2017) — scaled dot-product.
- `Rapport d'implémentation des mécanismes d'attention dans EnerGNN.pdf`, chapitre 14 — narrative complète de l'Approche 4 et spec littéral de l'Item 4.


# Partie 1 — Expériences sur LinearSystem

Performer vs LocalSum sur le DC power flow toy. Substrat minimal pour valider le pipeline et l'équivalence Form-A / Form-B (kernel-trick). La mesure de capacité se fait en partie 2 sur AC LF.


## 3.1. Construction de `LinearSystemProblemLoader` et helper

Loader identique aux Approches 0 et 1 (mêmes seed et configuration dataset) pour aligner les valeurs d'évaluation. Le helper `build_gnn` paramètre la message function : passer `LocalSumMessagePassingFunction` ou `PerformerMessagePassingFunction` permet une comparaison directe sur le même pipeline.


In [2]:
from energnn.model import (
    GNN, MLP, MLPEncoder, MLPEquivariantDecoder, RecurrentCoupler,
    PerformerMessagePassingFunction, LocalSumMessagePassingFunction,
    TDigestNormalizer,
)
from energnn.problem.example import LinearSystemProblemLoader
from energnn.trainer import Trainer

LATENT_DIM = 4  # Tiny config

ls_train = LinearSystemProblemLoader(seed=0)
ls_val = LinearSystemProblemLoader(seed=42)


def build_gnn(in_struct, out_struct, *, message_fn_cls, rngs):
    """Build a Tiny GNN whose message function is `message_fn_cls`.

    `message_fn_cls` is either LocalSumMessagePassingFunction or
    PerformerMessagePassingFunction. Both share the relevant subset of
    constructor surface (in_graph_structure, in_array_size, hidden_sizes,
    activation, out_size, use_bias, final_activation, outer_activation, rngs).
    LocalSum's additional encoded_feature_size is accepted via kwargs filter.
    """
    msg_kwargs = dict(
        in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh, rngs=rngs,
    )
    if message_fn_cls is LocalSumMessagePassingFunction:
        msg_kwargs["encoded_feature_size"] = LATENT_DIM
    msg_fn = message_fn_cls(**msg_kwargs)
    return GNN(
        normalizer=TDigestNormalizer(in_structure=in_struct, n_breakpoints=10, update_limit=1000),
        encoder=MLPEncoder(
            in_structure=in_struct, hidden_sizes=[], activation=nnx.leaky_relu,
            out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs,
        ),
        coupler=RecurrentCoupler(
            phi=MLP(
                in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
                out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs,
            ),
            message_functions=[msg_fn],
            n_steps=4,
        ),
        decoder=MLPEquivariantDecoder(
            in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
            activation=nnx.leaky_relu, out_structure=out_struct, use_bias=True,
            final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs,
        ),
    )


def count_params(model):
    _, params, _ = nnx.split(model, nnx.Param, ...)
    return sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(params) if hasattr(leaf, "size"))


print(f"train loader: {type(ls_train).__name__}, seed=0")
print(f"val loader:   {type(ls_val).__name__}, seed=42")
print(f"build_gnn(...) helper ready")


train loader: LinearSystemProblemLoader, seed=0
val loader:   LinearSystemProblemLoader, seed=42
build_gnn(...) helper ready


## 3.2. Form A (naïve) vs Form B (kernel-trick) parity check

Vérification numérique clé du Performer: les deux formes (sommation per-récepteur et factorisation outer-product) doivent produire la même sortie à la précision flottante près. Garde-fou du einsum: une transposition ou une permutation d'indice ferait diverger les deux formes.


In [3]:
# Verify Form A (naive O(n^2) per-receiver sum) and Form B (kernel-trick used
# in the implementation) produce numerically identical outputs.
import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx

from energnn.model import PerformerMessagePassingFunction
from energnn.problem.example import LinearSystemProblemLoader

ls_loader = LinearSystemProblemLoader(seed=0, batch_size=4, n_max=10)
ls_batch = next(iter(ls_loader))
ls_ctx_batch, _ = ls_batch.get_context()
ls_ctx = jax.tree.map(lambda x: x[0], ls_ctx_batch)
n_addr = int(ls_ctx.non_fictitious_addresses.shape[0])

mf = PerformerMessagePassingFunction(
    in_graph_structure=ls_loader.context_structure,
    in_array_size=4,
    hidden_sizes=[4],
    d_qk=8,
    out_size=3,
    activation=nnx.leaky_relu,
    outer_activation=nnx.identity,  # to read raw aggregator
    seed=11,
)

coords = jnp.array(np.random.RandomState(0).normal(size=(n_addr, 4)).astype(np.float32))
out_B, _ = mf(graph=ls_ctx, coordinates=coords)  # Form B used in impl

# Form A: per-receiver naive sum
q = mf.query_mlp(coords)
k = mf.key_mlp(coords)
v = mf.value_mlp(coords)
mask = jnp.expand_dims(ls_ctx.non_fictitious_addresses, -1)
k_m = k * mask
v_m = v * mask
scale = 1.0 / float(jnp.sqrt(jnp.float32(8)))

out_A = jnp.zeros((n_addr, 3), dtype=jnp.float32)
for a in range(n_addr):
    scores = jnp.einsum('nd,d->n', k_m, q[a])
    out_a = jnp.einsum('nd,n->d', v_m, scores) * scale
    out_A = out_A.at[a].set(out_a)

max_diff = float(jnp.max(jnp.abs(out_A - out_B)))
print(f'max |Form A - Form B| = {max_diff:.3e}  (expected < 1e-4)')
assert max_diff < 1e-4, 'kernel-trick form diverges from naive form'
print('parity OK')


max |Form A - Form B| = 9.537e-07  (expected < 1e-4)
parity OK


## 3.3. Entraînement de Performer vs LocalSum sur LinearSystem (5 epochs)

Deux GNN identiques à la message function près, entraînés sur les mêmes données, le même seed et la même configuration, évalués après chaque epoch.

LinearSystem est trop petit (3-4 classes) pour discriminer les mécanismes ; la comparaison de capacité se fait en partie 2 sur AC LF.


In [4]:
def train_and_eval(gnn, train_loader, val_loader, n_epochs):
    trainer = Trainer(model=gnn, gradient_transformation=optax.adam(1e-3))
    init, _ = trainer.eval(val_loader, progress_bar=False)
    curve = [float(init)]
    for _ in range(n_epochs):
        trainer.train(train_loader=train_loader, val_loader=None, n_epochs=1,
                      progress_bar=False, eval_before_training=False, eval_after_epoch=False)
        s, _ = trainer.eval(val_loader, progress_bar=False)
        curve.append(float(s))
    return curve


N_EPOCHS_LS = 5
ls_curves = {}
for label, mf_cls in (("LocalSum",          LocalSumMessagePassingFunction),
                     ("Performer", PerformerMessagePassingFunction)):
    gnn = build_gnn(ls_train.context_structure, ls_train.decision_structure,
                    message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
    n_params = count_params(gnn)
    ls_curves[label] = train_and_eval(gnn, ls_train, ls_val, N_EPOCHS_LS)
    print(f"{label:<20s} n_params={n_params:>5d}  init={ls_curves[label][0]:.3e}  final={ls_curves[label][-1]:.3e}")

print(f"\n{'epoch':<8s} {'LocalSum':>14s} {'Performer':>14s} {'Δ vs LocalSum':>16s}")
print("-" * 60)
for ep, (l, g) in enumerate(zip(ls_curves["LocalSum"], ls_curves["Performer"])):
    label = "init" if ep == 0 else f"ep {ep}"
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{label:<8s} {l:>14.4e} {g:>14.4e} {delta:>14.1f}%")


LocalSum             n_params=  185  init=4.981e-01  final=3.925e-01


Performer            n_params=  145  init=4.818e-01  final=4.473e-01

epoch          LocalSum      Performer    Δ vs LocalSum
------------------------------------------------------------
init         4.9811e-01     4.8180e-01           -3.3%
ep 1         4.7210e-01     4.7653e-01            0.9%
ep 2         4.4839e-01     4.6875e-01            4.5%
ep 3         4.2734e-01     4.6142e-01            8.0%
ep 4         4.0902e-01     4.5438e-01           11.1%
ep 5         3.9254e-01     4.4729e-01           13.9%


# Partie 2 — Expériences sur IEEE-9 et IEEE-14 (AC load flow)

Substrat de capacité : 11 classes d'hyper-arêtes, AC LF non linéaire, oracle V_mag / P / Q / I issu de pypowsybl. Le notebook se limite à 3 epochs Tiny ; le Gate 5 complet (Tiny + Small × 3 seeds × 15 epochs) est consigné dans `benchmarks/results/04_performer/baseline_performer_ac_load_flow.json`.


## 4.1. Construction de `ACLoadFlowProblemLoader` pour ieee9 et ieee14

Loader avec pré-cache (infrastructure partagée, commit `7d5f9f8` introduit à l'Approche 1) : 32 instances générées une seule fois dans `__init__`, réutilisées à chaque epoch. Même seed → mêmes données d'entraînement que les baselines des Approches 0 et 1, ce qui retire le drift en virgule flottante du training data lorsqu'on compare les deltas entre Approches.


In [5]:
import time
from ac_load_flow_problem import ACLoadFlowProblemLoader

LOADERS = {}
for net in ("ieee9", "ieee14"):
    t0 = time.perf_counter()
    LOADERS[net] = {
        "train": ACLoadFlowProblemLoader(network_name=net, dataset_size=32, batch_size=4,
                                          seed=0, perturbation_scale=0.1),
        "val":   ACLoadFlowProblemLoader(network_name=net, dataset_size=16, batch_size=4,
                                          seed=42, perturbation_scale=0.1),
    }
    print(f"{net}: built train+val loaders in {time.perf_counter() - t0:.1f}s")


ieee9: built train+val loaders in 6.2s


ieee14: built train+val loaders in 6.7s


## 4.2. Entraînement de Performer vs LocalSum sur ieee9 et ieee14 (3 epochs)

3 epochs Tiny par couple (réseau, message_fn). Sanity-check du pipeline Performer sur AC LF ; chiffres canoniques (3 seeds × Tiny/Small × 10-15 epochs) dans `BASELINES.md` section Approche 4.


In [6]:
N_EPOCHS_IEEE = 3
ieee_results = {}

for net in ("ieee9", "ieee14"):
    train_loader = LOADERS[net]["train"]
    val_loader = LOADERS[net]["val"]
    ieee_results[net] = {}
    for label, mf_cls in (("LocalSum",          LocalSumMessagePassingFunction),
                          ("Performer", PerformerMessagePassingFunction)):
        gnn = build_gnn(train_loader.context_structure, train_loader.decision_structure,
                        message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
        t0 = time.perf_counter()
        curve = train_and_eval(gnn, train_loader, val_loader, N_EPOCHS_IEEE)
        ieee_results[net][label] = {
            "curve": curve, "n_params": count_params(gnn),
            "train_time_s": time.perf_counter() - t0,
        }

print(f"{'network':<8s} {'method':<22s} {'n_params':>9s} {'init':>12s} {'final':>12s} {'train (s)':>11s}")
print("-" * 80)
for net in ("ieee9", "ieee14"):
    for label in ("LocalSum", "Performer"):
        r = ieee_results[net][label]
        print(f"{net:<8s} {label:<22s} {r['n_params']:>9d} {r['curve'][0]:>12.3e} "
              f"{r['curve'][-1]:>12.3e} {r['train_time_s']:>11.1f}")

print(f"\n{'network':<8s} {'LocalSum final':>16s} {'Performer final':>17s} {'Δ vs LocalSum':>16s}")
print("-" * 65)
for net in ("ieee9", "ieee14"):
    l = ieee_results[net]["LocalSum"]["curve"][-1]
    g = ieee_results[net]["Performer"]["curve"][-1]
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{net:<8s} {l:>16.4e} {g:>17.4e} {delta:>14.1f}%")


network  method                  n_params         init        final   train (s)
--------------------------------------------------------------------------------
ieee9    LocalSum                    1587    7.012e-01    5.140e-01        97.1
ieee9    Performer                    799    5.540e-01    5.015e-01        76.7
ieee14   LocalSum                    1587    3.817e-01    2.799e-01       125.7
ieee14   Performer                    799    3.604e-01    3.053e-01       105.5

network    LocalSum final   Performer final    Δ vs LocalSum
-----------------------------------------------------------------
ieee9          5.1402e-01        5.0149e-01           -2.4%
ieee14         2.7994e-01        3.0534e-01            9.1%


## 4.3. Gate 6 — surcoût Performer vs LocalSum

Médiane du forward et du forward+backward wall-time des deux message functions isolées sur le contexte ieee14. Mêmes hyper-paramètres, JIT-compilé, 20 warm-up + 100 mesures.

**Observation.** Performer implémente trois MLPs (Q, K, V) plus l'accumulation kernel-trick $M = \sum_{a'} V_{a'} K_{a'}^\top$ — coût $\mathcal{O}(n_{\mathrm{addr}} \cdot d_V \cdot d_{QK})$, quasi constant en `n_addr`. LocalSum croît linéairement avec le nombre d'hyper-arêtes per-(class, port). Sur les grands graphes (ieee118, ieee300), Performer devient plus rapide que LocalSum (ratio ×0,21 forward). Les chiffres complets Gate 6 (LinearSystem + ieee118 + ieee300) sont dans `benchmarks/results/04_performer/perf_performer_vs_localsum.json`.


In [7]:
import statistics

ieee14_loader = LOADERS["ieee14"]["train"]
ctx14, _ = ieee14_loader._cached_pairs[0]
n_addr14 = int(ctx14.non_fictitious_addresses.shape[0])
perf_coords = jnp.full((n_addr14, 4), 0.5, dtype=jnp.float32)

mf_perf_ls = LocalSumMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
mf_perf_ga = PerformerMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)


def make_jit_funcs(mf):
    @nnx.jit
    def fwd(m, c):
        out, _ = m(graph=ctx14, coordinates=c, get_info=False)
        return out

    @nnx.jit
    def fb(m, c):
        def loss(mod):
            out, _ = mod(graph=ctx14, coordinates=c, get_info=False)
            return jnp.mean(out ** 2)
        return nnx.value_and_grad(loss)(m)
    return fwd, fb


def time_calls(callable_, n_warmup=20, n_measure=100):
    for _ in range(n_warmup):
        out = callable_(); jax.block_until_ready(out)
    timings = []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        out = callable_(); jax.block_until_ready(out)
        timings.append(time.perf_counter() - t0)
    return statistics.median(timings) * 1000.0


fwd_ls, fb_ls = make_jit_funcs(mf_perf_ls)
fwd_ga, fb_ga = make_jit_funcs(mf_perf_ga)

ls_fwd_ms = time_calls(lambda: fwd_ls(mf_perf_ls, perf_coords))
ga_fwd_ms = time_calls(lambda: fwd_ga(mf_perf_ga, perf_coords))
ls_fb_ms = time_calls(lambda: fb_ls(mf_perf_ls, perf_coords))
ga_fb_ms = time_calls(lambda: fb_ga(mf_perf_ga, perf_coords))

print(f"{'':18s} {'LocalSum':>10s} {'Performer':>12s} {'overhead':>10s}")
print("-" * 56)
print(f"{'forward (ms)':18s} {ls_fwd_ms:>10.3f} {ga_fwd_ms:>12.3f} {ga_fwd_ms / ls_fwd_ms:>9.2f}x")
print(f"{'fwd+bwd (ms)':18s} {ls_fb_ms:>10.3f} {ga_fb_ms:>12.3f} {ga_fb_ms / ls_fb_ms:>9.2f}x")


                     LocalSum    Performer   overhead
--------------------------------------------------------
forward (ms)            7.853        1.773      0.23x
fwd+bwd (ms)            9.958        2.064      0.21x


## 4.4. Gate 7 — hash de reproductibilité in-process

L'output forward de Performer avec un seed fixé sur le contexte IEEE-14 (chargé depuis le cache picklé `benchmarks/results/04_performer/consistency_performer_context.pkl`) doit produire la même empreinte numérique entre re-runs in-process. Toute différence signalerait une régression de déterminisme.

Le Gate 7 officiel isole la reproductibilité de Performer en utilisant un cache de contexte picklé exécuté dans le script autonome (`benchmarks/04_performer/consistency_performer.py`). Le hash de référence est consigné dans `benchmarks/results/04_performer/consistency_performer.json`. La bit-identité cross-process se vérifie via ce script et non depuis le notebook (cache JIT et scheduling GPU varient).


In [8]:
import hashlib

# Run the standalone consistency script once to ensure the context cache exists.
# Then load the same cache and verify in-process forward determinism.
cache_path = _root / "benchmarks/results/04_performer/consistency_performer_context.pkl"
if not cache_path.exists():
    # Trigger consistency script which builds the cache.
    import subprocess
    print("Building consistency context cache (first run only)...")
    subprocess.run(
        ["python", str(_root / "benchmarks/04_performer/consistency_performer.py")],
        cwd=str(_root), check=True,
    )

import pickle
with cache_path.open("rb") as f:
    blob = pickle.load(f)
fixed_context = blob["context"]
fixed_structure = blob["structure"]

mf_fixed = PerformerMessagePassingFunction(
    in_graph_structure=fixed_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
n_addr_fixed = int(fixed_context.non_fictitious_addresses.shape[0])
fixed_coords = jnp.full((n_addr_fixed, 4), 0.5, dtype=jnp.float32)


def _forward_hash():
    out, _ = mf_fixed(graph=fixed_context, coordinates=fixed_coords, get_info=False)
    return hashlib.sha256(np.asarray(out, dtype=np.float64).tobytes()).hexdigest()


hash_a = _forward_hash()
hash_b = _forward_hash()

print(f"observed hash, call 1: {hash_a}")
print(f"observed hash, call 2: {hash_b}")
print(f"in-process match:      {hash_a == hash_b}")
assert hash_a == hash_b, "forward not in-process deterministic"
print("\nPASS: forward in-process deterministic (same hash across calls within same kernel).")
print("Cross-process bit-identical reference verified by")
print("  uv run python benchmarks/04_performer/consistency_performer.py")
print("(reference hash in benchmarks/results/04_performer/consistency_performer.json).")


observed hash, call 1: 0bf480ba5b1f0508dd4218ae0c8c697a5987813b21fa71832ab15cf204318161
observed hash, call 2: 0bf480ba5b1f0508dd4218ae0c8c697a5987813b21fa71832ab15cf204318161
in-process match:      True

PASS: forward in-process deterministic (same hash across calls within same kernel).
Cross-process bit-identical reference verified by
  uv run python benchmarks/04_performer/consistency_performer.py
(reference hash in benchmarks/results/04_performer/consistency_performer.json).


## 5. Synthèse et références

Résumé des observations de ce notebook (Approche 4, Item 4 `PerformerMessagePassingFunction`), structure alignée sur les notebooks 01 à 03.

Composantes vérifiées :

- **Section 2 — Spec et formes**: équivalence des deux formes du backlog (Form A somme per-récepteur, Form B kernel-trick) montrée analytiquement.
- **Section 3.2 — Form A vs Form B parity**: vérification numérique que la factorisation $M = \sum_{a'} V_{a'} K_{a'}^\top$ utilisée dans le `__call__` produit le même résultat que la somme naïve per-récepteur (à $10^{-4}$ près).
- **Section 3.3 — LinearSystem (5 epochs Tiny)** : entraînement court sur loader synthétique DC, comparé à LocalSum. Les runs Tiny + Small × 3 seeds sont dans `BASELINES.md` section Approche 4.
- **Section 4.2 — IEEE-9 et IEEE-14 (3 epochs Tiny)** : entraînement AC load flow sur deux substrats topologiques, comparé à LocalSum. Les runs Tiny + Small × 3 seeds sont dans `BASELINES.md` section Approche 4.
- **Section 4.3 — Perf (Gate 6)**: forward et fwd+bwd timing comparés à LocalSum sur trois substrats (LinearSystem, IEEE-118, IEEE-300).
- **Section 4.4 — Gate 7 reproductibilité**: empreinte numérique sur l'output forward, bit-identique entre re-runs.

Mesures Gate 6 (extraites de `benchmarks/results/04_performer/perf_performer_vs_localsum.json` ; les lignes LinearSystem / IEEE-118 / IEEE-300 ne sont pas reproduites par les cellules de ce notebook, qui mesure uniquement ieee14) :

| Substrat       | LocalSum fwd | Performer fwd | Ratio fwd | Performer fwd+bwd | Ratio fwd+bwd |
|----------------|:------------:|:-------------:|:---------:|:-----------------:|:-------------:|
| LinearSystem   | 2.165 ms     | 1.720 ms      | ×0.79     | 1.957 ms          | ×0.97         |
| IEEE-118       | 7.986 ms     | 1.662 ms      | ×0.21     | 1.975 ms          | ×0.19         |
| IEEE-300       | 8.014 ms     | 1.696 ms      | ×0.21     | 2.004 ms          | ×0.19         |

Le coût par forward du Performer est quasiment indépendant de la taille du substrat (≈1.7 ms forward, ≈2 ms fwd+bwd partout), conséquence directe de la forme kernel-trick $\mathcal{O}(n_{\mathrm{addr}} \cdot d_V \cdot d_{QK})$ avec $d_V, d_{QK}$ petits et constants. LocalSum scale linéairement en nombre d'hyper-edges, d'où le ratio ×0.21 sur IEEE-118/300.

Les chiffres d'entraînement Tiny/Small sur LinearSystem et IEEE-9/14 sont dans `BASELINES.md` section Approche 4.

**Références**:

- Vaswani et al. *Attention Is All You Need*. NeurIPS 2017 — scaled dot-product attention, convention du facteur $1/\sqrt{d_{QK}}$.
- Katharopoulos et al. *Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention*. ICML 2020 — forme sans softmax, base du kernel-trick utilisé ici.
- `Rapport d'implémentation des mécanismes d'attention dans EnerGNN.pdf`, chapitre 14 — narrative complète de l'Approche 4.
- `tests/model/unit/test_performer_message_passing.py`, `tests/model/integration/test_coupler.py`.
- `benchmarks/04_performer/`, `benchmarks/results/04_performer/`.
- `BASELINES.md` section Approche 4 (clôture).

---

*L'Approche 4 clôt l'Item 4 du backlog attention.*
